# Walmart Business Questions | My SQL Solutions

Each section has your saved SQL solution, followed by its output and a space for your own conclusion.

In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from IPython.display import display, HTML

load_dotenv()
engine = create_engine(os.environ['DATABASE_URL'])

def run_sql(sql):
    with engine.connect() as connection:
        result = connection.execute(text(sql))
        data = pd.DataFrame(result.fetchall(), columns=result.keys())
    display(HTML('''<style>.sql-results table{border-collapse:collapse;color:#f9fafb}.sql-results th{background:#1f2937;padding:8px}.sql-results td{background:#111827;padding:8px}.sql-results th,.sql-results td{border:1px solid #374151}</style><div class="sql-results">''' + data.to_html(index=False) + '</div>'))

print('Connected to PostgreSQL. Run any question below.')

Connected to PostgreSQL. Run any question below.


## Business Question 1
Find payment methods, transaction count, and quantities sold.

In [3]:
sql = '''
select payment_method, count(invoice_id) as no_of_transaction , sum(quantity) as total_quantity from walmart
group by payment_method;
'''
run_sql(sql)

payment_method,no_of_transaction,total_quantity
Credit card,4256,9567.0
Ewallet,3881,8932.0
Cash,1832,4984.0


**My notes / conclusion:** 


### Analysis

- Credit Card has the highest number of transactions (4,256), followed by Ewallet (3,881) and Cash (1,832).
- Credit Card also has the highest total quantity sold (9,567), followed by Ewallet (8,932) and Cash (4,984).
- Cash has the lowest transaction volume but the highest average quantity per transaction.

### Business Insight

The results indicate a strong preference for cashless payment methods, with Credit Card and Ewallet accounting for most transactions. Although Cash has fewer transactions, customers using Cash purchase more items per transaction on average (approximately 2.72 items).

### Limitation

The dataset shows which payment methods are used but does not explain why customers prefer a particular payment method. Therefore, reasons such as convenience, rewards, or credit availability should be treated as possible explanations rather than confirmed findings.

## Business Question 2
Find the highest-rated category in each branch.

In [4]:
sql = '''
SELECT *
FROM (
    SELECT
        branch,
        category,
        ROUND(CAST(AVG(rating) AS numeric), 2) AS avg_rating,
        ROW_NUMBER() OVER (
            PARTITION BY branch
            ORDER BY AVG(rating) DESC
        ) AS rown
    FROM walmart
    GROUP BY branch, category
) t
WHERE rown = 1
LIMIT 10;
'''
run_sql(sql)

branch,category,avg_rating,rown
WALM001,Electronic accessories,7.45,1
WALM002,Food and beverages,8.25,1
WALM003,Sports and travel,7.50,1
WALM004,Food and beverages,9.30,1
WALM005,Health and beauty,8.37,1
WALM006,Fashion accessories,6.80,1
WALM007,Food and beverages,7.55,1
WALM008,Food and beverages,7.40,1
WALM009,Sports and travel,9.60,1
WALM010,Electronic accessories,9.00,1


**My notes / conclusion:** 


### Analysis

- The highest-rated category varies across branches, with Health and beauty, Sports and travel, Food and beverages, and Electronic accessories appearing frequently as the top-rated categories.
- WALM034 has the highest branch-level average rating at 10.00 for Health and beauty.
- The lowest average rating among the selected top-rated categories is 5.28 for Electronic accessories at WALM050.
- This shows that customer ratings differ considerably across branches and their leading categories.

### Business Insight

Customer satisfaction varies by branch and category. Health and beauty, Sports and travel, Food and beverages, and Electronic accessories are frequently the highest-rated categories across branches. This suggests that customer satisfaction is not uniform across the store network and may vary based on the branch-category combination.

### Limitation

This analysis only identifies the highest-rated category within each branch. It does not establish the overall best category across Walmart or explain why ratings differ between branches. Additional product-level and customer-feedback data would be required to investigate the reasons behind these differences.

## Business Question 3
Find the busiest day for each branch.

In [5]:
sql = '''
select * from(
SELECT 
branch ,
to_char(date,'day') as day_name,
count(invoice_id) as total_transaction,
row_number() over(partition by branch order by count(invoice_id)desc) as rown
from walmart
group by branch,to_char(date,'day')
)t
where rown =1;
'''
run_sql(sql)

branch,day_name,total_transaction,rown
WALM001,thursday,16,1
WALM002,thursday,15,1
WALM003,tuesday,33,1
WALM004,sunday,14,1
WALM005,wednesday,19,1
WALM006,thursday,15,1
WALM007,sunday,12,1
WALM008,tuesday,17,1
WALM009,sunday,42,1
WALM010,wednesday,12,1


**My notes / conclusion:** 


### Analysis

- The busiest day varies across branches rather than being the same for every branch.
- Wednesday, Thursday, Tuesday, Sunday, Saturday, Friday, and Monday each appear as the busiest day for different branches.
- WALM058 has the highest transaction count among the branch-level busiest days, with 45 transactions on Wednesday.
- Other high-transaction branch-day combinations include WALM009 on Sunday (42 transactions), WALM069 on Thursday (42), WALM074 on Wednesday (41), and WALM030 on Wednesday (40).

### Business Insight

Transaction activity varies by branch and day of the week. This suggests that customer traffic patterns are not uniform across branches. The business could use branch-level day patterns to plan staffing, inventory, and promotions according to expected transaction activity.

### Limitation

This analysis identifies the busiest day for each branch but does not explain why transaction volume is higher on those days. Additional information such as sales value, promotions, holidays, or customer traffic would be needed to identify the underlying reasons.

## Business Question 4
Find total quantity sold per payment method.

In [6]:
sql = '''
select payment_method,sum(quantity) as quantity_sold from walmart
group by payment_method
'''
run_sql(sql)

payment_method,quantity_sold
Credit card,9567.0
Ewallet,8932.0
Cash,4984.0


**My notes / conclusion:** 


### Analysis

- Credit Card has the highest quantity sold (9,567).
- Ewallet is the second-highest with 8,932 items sold.
- Cash has the lowest quantity sold with 4,984 items.
- Credit Card accounts for 635 more items sold than Ewallet.

### Business Insight

Credit Card is the leading payment method by quantity sold, followed closely by Ewallet, while Cash accounts for the lowest quantity sold. This indicates that cashless payment methods contribute substantially to the overall quantity of items sold.

### Limitation

This analysis only compares quantity sold by payment method. It does not explain why customers choose a particular payment method or whether the payment method affects transaction value.

## Business Question 5
Find average, minimum, and maximum rating per city.

In [7]:
sql = '''
SELECT city, 
       ROUND(CAST(AVG(rating) AS numeric), 2) AS avg_rating, 
       MIN(rating) AS min_rating, 
       MAX(rating) AS max_rating 
FROM walmart
GROUP BY city;
'''
run_sql(sql)

city,avg_rating,min_rating,max_rating
Flower Mound,6.47,4.0,9.8
College Station,6.67,4.0,10.0
McKinney,6.09,3.0,9.0
Nacogdoches,6.40,3.0,9.1
Laredo,6.64,3.0,9.6
Huntsville,6.81,4.0,9.7
Arlington,6.22,3.0,9.6
Bryan,6.61,4.0,10.0
New Braunfels,6.27,3.0,9.6
DeSoto,6.24,3.0,9.9


**My notes / conclusion:** 


### Analysis

- Austin has the highest average rating among the listed cities at 7.00.
- Denton (6.68), Pflugerville (6.73), Huntsville (6.81), and College Station (6.67) also have relatively high average ratings.
- Rowlett has the lowest average rating at 4.99, followed by Texas City at 5.04 and San Marcos at 5.08.
- Several cities have a maximum rating of 10.00, showing that highly positive customer experiences exist even in cities with lower average ratings.
- Minimum ratings are mostly between 3 and 4, showing that some customers reported relatively low satisfaction.

### Business Insight

Customer satisfaction varies considerably across cities. Austin has the strongest overall average rating, while cities such as Rowlett, Texas City, and San Marcos have comparatively lower average ratings. These lower-rated cities may require further investigation to understand the factors affecting customer satisfaction.

### Limitation

The analysis only provides average, minimum, and maximum ratings by city. It does not explain why ratings differ between cities. In addition, minimum and maximum ratings alone do not show how frequently those ratings occurred, so the distribution of individual ratings would provide deeper insight.

## Business Question 6
Find total profit per category.

In [8]:
sql = '''
SELECT 
    category, 
    ROUND(cast(SUM(unit_price * quantity * profit_margin)as numeric),3) AS total_profit
FROM walmart
GROUP BY category
ORDER BY total_profit DESC;
'''
run_sql(sql)

category,total_profit
Fashion accessories,192314.893
Home and lifestyle,192213.638
Electronic accessories,30772.490
Food and beverages,21552.862
Sports and travel,20613.808
Health and beauty,18671.735


**My notes / conclusion:** 


### Analysis

- Fashion accessories has the highest total profit at 192,314.893.
- Home and lifestyle is a very close second with 192,213.638.
- These two categories generate substantially higher profit than the remaining categories.
- Health and beauty has the lowest total profit at 18,671.735.
- Fashion accessories generates approximately 10.3 times the profit of Health and beauty.

### Business Insight

Fashion accessories and Home and lifestyle are the strongest categories in terms of total profit, while Health and beauty generates the lowest total profit. Interestingly, Health and beauty was frequently the highest-rated category across branches in Question 2, showing that customer satisfaction and profitability do not necessarily move together.

### Limitation

This analysis calculates total profit using the given formula but does not explain why profitability differs between categories. Additional analysis of sales volume, unit prices, and profit margins would be required to identify the factors driving the differences.

## Business Question 7
Find preferred payment method per branch.

In [9]:
sql = '''
select branch,payment_method from (
SELECT *,
       ROW_NUMBER() OVER (
           PARTITION BY branch
           ORDER BY transac_pb DESC
       ) AS rn
FROM (
    SELECT 
        branch,
        payment_method,
        COUNT(payment_method) AS transac_pb
    FROM walmart
    GROUP BY branch, payment_method
) t
) x
where rn=1;
'''
run_sql(sql)

branch,payment_method
WALM001,Ewallet
WALM002,Ewallet
WALM003,Credit card
WALM004,Ewallet
WALM005,Ewallet
WALM006,Ewallet
WALM007,Ewallet
WALM008,Ewallet
WALM009,Credit card
WALM010,Ewallet


**My notes / conclusion:** 


### Analysis

- Ewallet is the preferred payment method in 75 out of 100 branches.
- Credit card is the preferred method in 23 branches.
- Cash is the preferred payment method in only 2 branches: WALM074 and WALM082.
- Ewallet is therefore the dominant preferred payment method across individual branches.

### Business Insight

Ewallet is the most commonly preferred payment method across the branch network, indicating strong adoption of digital wallet payments at the branch level. However, Question 1 showed that Credit Card had the highest overall transaction volume. This difference suggests that branch-level preference and overall transaction volume can tell different stories and should be analysed separately.

### Limitation

The analysis identifies the most frequently used payment method within each branch but does not explain why customers prefer a particular method. The difference between branch-level preference and overall transaction volume would require further branch-level transaction analysis to understand.

## Business Question 8
Categorize transactions into morning, afternoon, and evening shifts.

In [10]:
sql = '''
 select
 case
   when time<'12:00:00' then 'morning'
   when time<'17:00:00' then 'afternoon'
   else  'Evening'
  End as shift,count (invoice_id) as no_trans
  from walmart
  group by shift
 order by no_trans desc
'''
run_sql(sql)

shift,no_trans
Evening,5546
afternoon,3609
morning,814


**My notes / conclusion:** 


### Analysis

- Evening has the highest transaction volume with 5,546 transactions.
- Afternoon has 3,609 transactions, while morning has only 814.
- Evening accounts for approximately 55.6% of all transactions.
- Afternoon accounts for approximately 36.2%, while morning accounts for approximately 8.2%.
- Approximately 91.8% of transactions occur during the afternoon and evening combined.

### Business Insight

Customer transaction activity is strongly concentrated in the afternoon and evening, with the evening shift accounting for the majority of transactions. Morning has substantially lower transaction activity. This pattern could help the business plan staffing and operational resources around periods of higher customer activity.

### Limitation

The analysis shows when transactions occur but does not explain why evening activity is higher. Factors such as customer schedules, store operating hours, promotions, or other external factors would require additional data to investigate.